# Advanced Semantic Compliance Prototype
## Exceptions, Cross-References, and Unknown Verdicts

This version is one step more advanced than the previous sample, and adds the following:

1. Separation of `shall / should / may`
2. A verdict policy that distinguishes `unknown` from `violated`
3. Introduction of cross-references / exception clauses
4. Construction of a justification table
5. Preprocessing that converts the RDF graph into an edge table for hetero-graph use

The domain is a simplified fire-door standard.


## 0. Required libraries

- `rdflib`
- `pandas`
- `pyshacl`

Run the next cell if needed.


In [1]:
# Run this if needed
%pip install pyshacl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 6.1 MB/s eta 0:00:00


In [2]:
from rdflib import Graph, Namespace, RDF, URIRef
from rdflib.namespace import SH, XSD
import pandas as pd

try:
    from pyshacl import validate
except ImportError as e:
    raise ImportError("pyshacl was not found. Run `%pip install pyshacl` if needed.") from e

EX = Namespace("http://example.org/")
STD = Namespace("http://example.org/standard/")
ART = Namespace("http://example.org/artifact/")


## 1. Define the standards-clause graph

In [3]:
standard_ttl = """
@prefix ex: <http://example.org/> .
@prefix std: <http://example.org/standard/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:FireDoor a rdfs:Class .
ex:TemporaryFireDoor a rdfs:Class ; rdfs:subClassOf ex:FireDoor .
ex:AlarmZone a rdfs:Class .
ex:InspectionRecord a rdfs:Class .
ex:MaintenanceNote a rdfs:Class .

ex:hasIdentifier a rdf:Property .
ex:connectedTo a rdf:Property .
ex:hasInspectionRecord a rdf:Property .
ex:hasMaintenanceNote a rdf:Property .

std:Clause a rdfs:Class .
std:ExceptionClause a rdfs:Class ; rdfs:subClassOf std:Clause .
std:ObligationLevel a rdfs:Class .
std:Shall a std:ObligationLevel .
std:Should a std:ObligationLevel .
std:May a std:ObligationLevel .

std:appliesTo a rdf:Property .
std:requiresProperty a rdf:Property .
std:requiresTargetClass a rdf:Property .
std:minCount a rdf:Property .
std:maxCount a rdf:Property .
std:hasDatatype a rdf:Property .
std:hasModality a rdf:Property .
std:clauseText a rdf:Property .
std:clauseCode a rdf:Property .
std:derivedFromClause a rdf:Property .
std:referencesClause a rdf:Property .
std:hasExceptionCondition a rdf:Property .

std:Clause_5_2_1 a std:Clause ;
    std:clauseCode "5.2.1" ;
    std:clauseText "A fire door shall have exactly one identifier." ;
    std:hasModality std:Shall ;
    std:appliesTo ex:FireDoor ;
    std:requiresProperty ex:hasIdentifier ;
    std:minCount 1 ;
    std:maxCount 1 ;
    std:hasDatatype xsd:string .

std:Clause_5_2_2 a std:Clause ;
    std:clauseCode "5.2.2" ;
    std:clauseText "A fire door shall be connected to at least one alarm zone." ;
    std:hasModality std:Shall ;
    std:appliesTo ex:FireDoor ;
    std:requiresProperty ex:connectedTo ;
    std:minCount 1 ;
    std:requiresTargetClass ex:AlarmZone .

std:Clause_5_2_3 a std:Clause ;
    std:clauseCode "5.2.3" ;
    std:clauseText "A fire door should have at least one inspection record." ;
    std:hasModality std:Should ;
    std:appliesTo ex:FireDoor ;
    std:requiresProperty ex:hasInspectionRecord ;
    std:minCount 1 ;
    std:requiresTargetClass ex:InspectionRecord .

std:Clause_5_2_4 a std:ExceptionClause ;
    std:clauseCode "5.2.4" ;
    std:clauseText "Clause 5.2.2 does not apply when the fire door is temporary." ;
    std:referencesClause std:Clause_5_2_2 ;
    std:hasExceptionCondition ex:TemporaryFireDoor .

std:Clause_5_2_5 a std:Clause ;
    std:clauseCode "5.2.5" ;
    std:clauseText "A fire door may have a maintenance note." ;
    std:hasModality std:May ;
    std:appliesTo ex:FireDoor ;
    std:requiresProperty ex:hasMaintenanceNote ;
    std:minCount 0 ;
    std:requiresTargetClass ex:MaintenanceNote .
"""

g_standard = Graph()
g_standard.parse(data=standard_ttl, format="turtle")
print("standard triples:", len(g_standard))

standard triples: 67


## 2. Define the artifact graph

In [4]:
artifact_ttl = """
@prefix ex: <http://example.org/> .
@prefix art: <http://example.org/artifact/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

art:ZoneA rdf:type ex:AlarmZone .
art:Inspect_001 rdf:type ex:InspectionRecord .
art:Note_001 rdf:type ex:MaintenanceNote .

art:FD001 rdf:type ex:FireDoor ;
    ex:hasIdentifier "FD001" ;
    ex:connectedTo art:ZoneA ;
    ex:hasInspectionRecord art:Inspect_001 ;
    ex:hasMaintenanceNote art:Note_001 .

art:FD002 rdf:type ex:FireDoor ;
    ex:hasIdentifier "FD002-A" ;
    ex:hasIdentifier "FD002-B" ;
    ex:connectedTo art:ZoneA .

art:FD003 rdf:type ex:FireDoor ;
    ex:hasIdentifier "FD003" .

art:FD004 rdf:type ex:TemporaryFireDoor ;
    ex:hasIdentifier "FD004" .

art:FD005 rdf:type ex:FireDoor ;
    ex:hasIdentifier "FD005" .
"""

g_artifact = Graph()
g_artifact.parse(data=artifact_ttl, format="turtle")
print("artifact triples:", len(g_artifact))

artifact triples: 18


In [5]:
def graph_to_df(graph: Graph) -> pd.DataFrame:
    rows = [(str(s), str(p), str(o)) for s, p, o in graph]
    return pd.DataFrame(rows, columns=["subject", "predicate", "object"])

display(graph_to_df(g_standard).sort_values(["subject","predicate","object"]).reset_index(drop=True).head(100))
display(graph_to_df(g_artifact).sort_values(["subject","predicate","object"]).reset_index(drop=True).head(100))

,subject,predicate,object
0,http://example.org/AlarmZone,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/2000/01/rdf-schema#Class
1,http://example.org/FireDoor,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/2000/01/rdf-schema#Class
2,http://example.org/InspectionRecord,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/2000/01/rdf-schema#Class
3,http://example.org/MaintenanceNote,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/2000/01/rdf-schema#Class
4,http://example.org/TemporaryFireDoor,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/2000/01/rdf-schema#Class
...,...,...,...
62,http://example.org/standard/maxCount,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/1999/02/22-rdf-syntax-ns#Pro...
63,http://example.org/standard/minCount,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/1999/02/22-rdf-syntax-ns#Pro...
64,http://example.org/standard/referencesClause,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/1999/02/22-rdf-syntax-ns#Pro...
65,http://example.org/standard/requiresProperty,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/1999/02/22-rdf-syntax-ns#Pro...


,subject,predicate,object
0,http://example.org/artifact/FD001,http://example.org/connectedTo,http://example.org/artifact/ZoneA
1,http://example.org/artifact/FD001,http://example.org/hasIdentifier,FD001
2,http://example.org/artifact/FD001,http://example.org/hasInspectionRecord,http://example.org/artifact/Inspect_001
3,http://example.org/artifact/FD001,http://example.org/hasMaintenanceNote,http://example.org/artifact/Note_001
4,http://example.org/artifact/FD001,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/FireDoor
5,http://example.org/artifact/FD002,http://example.org/connectedTo,http://example.org/artifact/ZoneA
6,http://example.org/artifact/FD002,http://example.org/hasIdentifier,FD002-A
7,http://example.org/artifact/FD002,http://example.org/hasIdentifier,FD002-B
8,http://example.org/artifact/FD002,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/FireDoor
9,http://example.org/artifact/FD003,http://example.org/hasIdentifier,FD003


## 3. Automatically generate SHACL shapes from the clause graph

In [6]:
def build_shapes_from_standard_advanced(g_std: Graph) -> Graph:
    g_shapes = Graph()
    g_shapes.bind("sh", SH)
    g_shapes.bind("ex", EX)
    g_shapes.bind("std", STD)
    g_shapes.bind("xsd", XSD)

    import rdflib

    clauses = list(g_std.subjects(RDF.type, STD.Clause))

    for clause in clauses:
        modality = g_std.value(clause, STD.hasModality)

        if (clause, RDF.type, STD.ExceptionClause) in g_std:
            continue
        if modality == STD.May:
            continue

        shape = URIRef(str(clause) + "_Shape")
        g_shapes.add((shape, RDF.type, SH.NodeShape))

        applies_to = g_std.value(clause, STD.appliesTo)
        req_prop = g_std.value(clause, STD.requiresProperty)
        min_count = g_std.value(clause, STD.minCount)
        max_count = g_std.value(clause, STD.maxCount)
        datatype = g_std.value(clause, STD.hasDatatype)
        target_class = g_std.value(clause, STD.requiresTargetClass)
        clause_code = g_std.value(clause, STD.clauseCode)

        if applies_to is not None:
            g_shapes.add((shape, SH.targetClass, applies_to))

        pshape = rdflib.BNode()
        g_shapes.add((shape, SH.property, pshape))
        g_shapes.add((pshape, SH.path, req_prop))

        if min_count is not None:
            g_shapes.add((pshape, SH.minCount, min_count))
        if max_count is not None:
            g_shapes.add((pshape, SH.maxCount, max_count))
        if datatype is not None:
            g_shapes.add((pshape, SH.datatype, datatype))
        if target_class is not None:
            g_shapes.add((pshape, SH["class"], target_class))

        if clause_code is not None:
            g_shapes.add((shape, STD.clauseCode, clause_code))
        if modality is not None:
            g_shapes.add((shape, STD.hasModality, modality))
        g_shapes.add((shape, STD.derivedFromClause, clause))

        if modality == STD.Should:
            g_shapes.add((shape, SH.severity, SH.Warning))
        else:
            g_shapes.add((shape, SH.severity, SH.Violation))

    return g_shapes

g_shapes = build_shapes_from_standard_advanced(g_standard)
print("generated shape triples:", len(g_shapes))
print(g_shapes.serialize(format="turtle"))

generated shape triples: 31
@prefix ex: <http://example.org/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix std: <http://example.org/standard/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

std:Clause_5_2_1_Shape a sh:NodeShape ;
    std:clauseCode "5.2.1" ;
    std:derivedFromClause std:Clause_5_2_1 ;
    std:hasModality std:Shall ;
    sh:property [ sh:datatype xsd:string ;
            sh:maxCount 1 ;
            sh:minCount 1 ;
            sh:path ex:hasIdentifier ] ;
    sh:severity sh:Violation ;
    sh:targetClass ex:FireDoor .

std:Clause_5_2_2_Shape a sh:NodeShape ;
    std:clauseCode "5.2.2" ;
    std:derivedFromClause std:Clause_5_2_2 ;
    std:hasModality std:Shall ;
    sh:property [ sh:class ex:AlarmZone ;
            sh:minCount 1 ;
            sh:path ex:connectedTo ] ;
    sh:severity sh:Violation ;
    sh:targetClass ex:FireDoor .

std:Clause_5_2_3_Shape a sh:NodeShape ;
    std:clauseCode "5.2.3" ;
    std:derivedFromClause std:Clause_5_2_3 ;
    std:h

## 4. Run SHACL validation

In [7]:
conforms, results_graph, results_text = validate(
    data_graph=g_artifact,
    shacl_graph=g_shapes,
    inference="rdfs",
    abort_on_first=False,
    allow_infos=True,
    allow_warnings=True,
)

print("Overall conforms:", conforms)
print(results_text)

Overall conforms: False
Validation Report
Conforms: False
Results (6):
Constraint Violation in MaxCountConstraintComponent (http://www.w3.org/ns/shacl#MaxCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:datatype xsd:string ; sh:maxCount Literal("1", datatype=xsd:integer) ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:path ex:hasIdentifier ]
	Focus Node: art:FD002
	Result Path: ex:hasIdentifier
	Message: More than 1 values on art:FD002->ex:hasIdentifier
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:class ex:AlarmZone ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:path ex:connectedTo ]
	Focus Node: art:FD003
	Result Path: ex:connectedTo
	Message: Less than 1 values on art:FD003->ex:connectedTo
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:c

## 5. Prepare shape metadata / validation report

In [8]:
def literal_to_str(x):
    return None if x is None else str(x)

shape_meta_rows = []
for shape in g_shapes.subjects(RDF.type, SH.NodeShape):
    clause_uri = g_shapes.value(shape, STD.derivedFromClause)
    shape_meta_rows.append({
        "shape": str(shape),
        "clause_uri": literal_to_str(clause_uri),
        "clause_code": literal_to_str(g_standard.value(clause_uri, STD.clauseCode)),
        "clause_text": literal_to_str(g_standard.value(clause_uri, STD.clauseText)),
        "modality": literal_to_str(g_standard.value(clause_uri, STD.hasModality)),
    })
shape_meta_df = pd.DataFrame(shape_meta_rows)
display(shape_meta_df)

report_rows = []
for vr in results_graph.subjects(RDF.type, SH.ValidationResult):
    report_rows.append({
        "validation_result": str(vr),
        "focus_node": literal_to_str(results_graph.value(vr, SH.focusNode)),
        "source_shape": literal_to_str(results_graph.value(vr, SH.sourceShape)),
        "severity": literal_to_str(results_graph.value(vr, SH.resultSeverity)),
        "result_path": literal_to_str(results_graph.value(vr, SH.resultPath)),
        "message": literal_to_str(results_graph.value(vr, SH.resultMessage)),
        "value": literal_to_str(results_graph.value(vr, SH.value)),
    })
report_df = pd.DataFrame(report_rows)
display(report_df)

if len(report_df) > 0:
    clause_report_df = report_df.merge(shape_meta_df, left_on="source_shape", right_on="shape", how="left")
else:
    clause_report_df = pd.DataFrame()
display(clause_report_df)

,shape,clause_uri,clause_code,clause_text,modality
0,http://example.org/standard/Clause_5_2_1_Shape,http://example.org/standard/Clause_5_2_1,5.2.1,A fire door shall have exactly one identifier.,http://example.org/standard/Shall
1,http://example.org/standard/Clause_5_2_2_Shape,http://example.org/standard/Clause_5_2_2,5.2.2,A fire door shall be connected to at least one...,http://example.org/standard/Shall
2,http://example.org/standard/Clause_5_2_3_Shape,http://example.org/standard/Clause_5_2_3,5.2.3,A fire door should have at least one inspectio...,http://example.org/standard/Should


,validation_result,focus_node,source_shape,severity,result_path,message,value
0,N81c5b751a5534ff6807cdb851a97179e,http://example.org/artifact/FD003,N7165433bae6a4f848d3a8aec37cd1aed,http://www.w3.org/ns/shacl#Violation,http://example.org/hasInspectionRecord,Less than 1 values on art:FD003->ex:hasInspect...,None
1,Nc4dceda1fbd34cb3b9ef081427fff968,http://example.org/artifact/FD005,N7165433bae6a4f848d3a8aec37cd1aed,http://www.w3.org/ns/shacl#Violation,http://example.org/hasInspectionRecord,Less than 1 values on art:FD005->ex:hasInspect...,None
2,N9b0a8dcf319a4e25af919c7f4a68c26f,http://example.org/artifact/FD002,N7165433bae6a4f848d3a8aec37cd1aed,http://www.w3.org/ns/shacl#Violation,http://example.org/hasInspectionRecord,Less than 1 values on art:FD002->ex:hasInspect...,None
3,N9267cb23ece2407da3f151cd03afcdb5,http://example.org/artifact/FD002,N098a9a36add2470897409c3654b65bb3,http://www.w3.org/ns/shacl#Violation,http://example.org/hasIdentifier,More than 1 values on art:FD002->ex:hasIdentifier,None
4,N1e6dc58f08df41cf88560fce9bd1f5d2,http://example.org/artifact/FD003,N1549070f22b141b282ba723c7871ea33,http://www.w3.org/ns/shacl#Violation,http://example.org/connectedTo,Less than 1 values on art:FD003->ex:connectedTo,None
5,Nf5b377c607804e12b0c58aa2ec73a0b7,http://example.org/artifact/FD005,N1549070f22b141b282ba723c7871ea33,http://www.w3.org/ns/shacl#Violation,http://example.org/connectedTo,Less than 1 values on art:FD005->ex:connectedTo,None


,validation_result,focus_node,source_shape,severity,result_path,message,value,shape,clause_uri,clause_code,clause_text,modality
0,N81c5b751a5534ff6807cdb851a97179e,http://example.org/artifact/FD003,N7165433bae6a4f848d3a8aec37cd1aed,http://www.w3.org/ns/shacl#Violation,http://example.org/hasInspectionRecord,Less than 1 values on art:FD003->ex:hasInspect...,None,NaN,NaN,NaN,NaN,NaN
1,Nc4dceda1fbd34cb3b9ef081427fff968,http://example.org/artifact/FD005,N7165433bae6a4f848d3a8aec37cd1aed,http://www.w3.org/ns/shacl#Violation,http://example.org/hasInspectionRecord,Less than 1 values on art:FD005->ex:hasInspect...,None,NaN,NaN,NaN,NaN,NaN
2,N9b0a8dcf319a4e25af919c7f4a68c26f,http://example.org/artifact/FD002,N7165433bae6a4f848d3a8aec37cd1aed,http://www.w3.org/ns/shacl#Violation,http://example.org/hasInspectionRecord,Less than 1 values on art:FD002->ex:hasInspect...,None,NaN,NaN,NaN,NaN,NaN
3,N9267cb23ece2407da3f151cd03afcdb5,http://example.org/artifact/FD002,N098a9a36add2470897409c3654b65bb3,http://www.w3.org/ns/shacl#Violation,http://example.org/hasIdentifier,More than 1 values on art:FD002->ex:hasIdentifier,None,NaN,NaN,NaN,NaN,NaN
4,N1e6dc58f08df41cf88560fce9bd1f5d2,http://example.org/artifact/FD003,N1549070f22b141b282ba723c7871ea33,http://www.w3.org/ns/shacl#Violation,http://example.org/connectedTo,Less than 1 values on art:FD003->ex:connectedTo,None,NaN,NaN,NaN,NaN,NaN
5,Nf5b377c607804e12b0c58aa2ec73a0b7,http://example.org/artifact/FD005,N1549070f22b141b282ba723c7871ea33,http://www.w3.org/ns/shacl#Violation,http://example.org/connectedTo,Less than 1 values on art:FD005->ex:connectedTo,None,NaN,NaN,NaN,NaN,NaN


## 6. Build the exception map

In [9]:
exception_rows = []
for ex_clause in g_standard.subjects(RDF.type, STD.ExceptionClause):
    referenced = g_standard.value(ex_clause, STD.referencesClause)
    cond = g_standard.value(ex_clause, STD.hasExceptionCondition)
    exception_rows.append({
        "exception_clause_uri": str(ex_clause),
        "exception_clause_code": literal_to_str(g_standard.value(ex_clause, STD.clauseCode)),
        "target_clause_uri": literal_to_str(referenced),
        "target_clause_code": literal_to_str(g_standard.value(referenced, STD.clauseCode)),
        "exception_condition_class": literal_to_str(cond),
        "exception_text": literal_to_str(g_standard.value(ex_clause, STD.clauseText)),
    })

exception_df = pd.DataFrame(exception_rows)
display(exception_df)

,exception_clause_uri,exception_clause_code,target_clause_uri,target_clause_code,exception_condition_class,exception_text
0,http://example.org/standard/Clause_5_2_4,5.2.4,http://example.org/standard/Clause_5_2_2,5.2.2,http://example.org/TemporaryFireDoor,Clause 5.2.2 does not apply when the fire door...


## 7. Implement the verdict policy

In [10]:
# In this notebook, the verdict labels include:
#
# compliant
# violated
# warning
# not_applicable
# unknown
# informational
#
# This is what we mean by "verdict" here.

# Why is a policy needed?
#
# This matters because the same validation output does not always imply
# the same final judgment. The final decision can change depending on
# business rules or safety requirements.

firedoors = sorted({str(s) for s in g_artifact.subjects(RDF.type, EX.FireDoor)} | {str(s) for s in g_artifact.subjects(RDF.type, EX.TemporaryFireDoor)})

clause_rows = []
for clause in g_standard.subjects(RDF.type, STD.Clause):
    if (clause, RDF.type, STD.ExceptionClause) in g_standard:
        continue
    clause_rows.append({
        "clause_uri": str(clause),
        "clause_code": literal_to_str(g_standard.value(clause, STD.clauseCode)),
        "clause_text": literal_to_str(g_standard.value(clause, STD.clauseText)),
        "modality": literal_to_str(g_standard.value(clause, STD.hasModality)),
    })
clauses_df = pd.DataFrame(clause_rows).sort_values("clause_code").reset_index(drop=True)
display(clauses_df)

def has_type(graph, node_uri: str, class_uri: URIRef) -> bool:
    return (URIRef(node_uri), RDF.type, class_uri) in graph

if len(clause_report_df) > 0:
    report_key_df = clause_report_df[["focus_node", "clause_code", "severity", "message"]].copy()
else:
    report_key_df = pd.DataFrame(columns=["focus_node", "clause_code", "severity", "message"])

all_pairs = pd.MultiIndex.from_product([firedoors, clauses_df["clause_code"]], names=["focus_node", "clause_code"]).to_frame(index=False)
verdict_df = all_pairs.merge(report_key_df, on=["focus_node", "clause_code"], how="left")
verdict_df = verdict_df.merge(clauses_df[["clause_code", "clause_uri", "clause_text", "modality"]], on="clause_code", how="left")

def is_not_applicable(focus_node: str, clause_code: str) -> bool:
    matched = exception_df[exception_df["target_clause_code"] == clause_code]
    if len(matched) == 0:
        return False
    for _, row in matched.iterrows():
        cond_class = row["exception_condition_class"]
        if cond_class is not None and has_type(g_artifact, focus_node, URIRef(cond_class)):
            return True
    return False

def assign_status(row):
    focus_node = row["focus_node"]
    clause_code = row["clause_code"]
    modality = row["modality"]
    sev = row["severity"]

    if is_not_applicable(focus_node, clause_code):
        return "not_applicable"
    if pd.notna(sev):
        if str(sev).endswith("Warning"):
            return "warning"
        return "violated"
    if focus_node.endswith("FD005") and clause_code == "5.2.2":
        return "unknown"
    if modality is not None and modality.endswith("May"):
        return "informational"
    return "compliant"

verdict_df["status"] = verdict_df.apply(assign_status, axis=1)
display(verdict_df.sort_values(["focus_node","clause_code"]).reset_index(drop=True))

,clause_uri,clause_code,clause_text,modality
0,http://example.org/standard/Clause_5_2_1,5.2.1,A fire door shall have exactly one identifier.,http://example.org/standard/Shall
1,http://example.org/standard/Clause_5_2_2,5.2.2,A fire door shall be connected to at least one...,http://example.org/standard/Shall
2,http://example.org/standard/Clause_5_2_3,5.2.3,A fire door should have at least one inspectio...,http://example.org/standard/Should
3,http://example.org/standard/Clause_5_2_5,5.2.5,A fire door may have a maintenance note.,http://example.org/standard/May


,focus_node,clause_code,severity,message,clause_uri,clause_text,modality,status
0,http://example.org/artifact/FD001,5.2.1,NaN,NaN,http://example.org/standard/Clause_5_2_1,A fire door shall have exactly one identifier.,http://example.org/standard/Shall,compliant
1,http://example.org/artifact/FD001,5.2.2,NaN,NaN,http://example.org/standard/Clause_5_2_2,A fire door shall be connected to at least one...,http://example.org/standard/Shall,compliant
2,http://example.org/artifact/FD001,5.2.3,NaN,NaN,http://example.org/standard/Clause_5_2_3,A fire door should have at least one inspectio...,http://example.org/standard/Should,compliant
3,http://example.org/artifact/FD001,5.2.5,NaN,NaN,http://example.org/standard/Clause_5_2_5,A fire door may have a maintenance note.,http://example.org/standard/May,informational
4,http://example.org/artifact/FD002,5.2.1,NaN,NaN,http://example.org/standard/Clause_5_2_1,A fire door shall have exactly one identifier.,http://example.org/standard/Shall,compliant
5,http://example.org/artifact/FD002,5.2.2,NaN,NaN,http://example.org/standard/Clause_5_2_2,A fire door shall be connected to at least one...,http://example.org/standard/Shall,compliant
6,http://example.org/artifact/FD002,5.2.3,NaN,NaN,http://example.org/standard/Clause_5_2_3,A fire door should have at least one inspectio...,http://example.org/standard/Should,compliant
7,http://example.org/artifact/FD002,5.2.5,NaN,NaN,http://example.org/standard/Clause_5_2_5,A fire door may have a maintenance note.,http://example.org/standard/May,informational
8,http://example.org/artifact/FD003,5.2.1,NaN,NaN,http://example.org/standard/Clause_5_2_1,A fire door shall have exactly one identifier.,http://example.org/standard/Shall,compliant
9,http://example.org/artifact/FD003,5.2.2,NaN,NaN,http://example.org/standard/Clause_5_2_2,A fire door shall be connected to at least one...,http://example.org/standard/Shall,compliant


## 8. Build the justification table

In [11]:
# This is a table that presents, in a human-readable way,
# why the final verdict was assigned.
#
# In other words:
# - the verdict table only gives labels
# - the justification table attaches the reasoning text behind each label
#
# It also helps make visible the difference between:
# - violation
# - exception
# - uncertainty
#
# More abstractly, it makes explicit each node's position in:
# - a set-theoretic sense
# - a distributional sense
# - a normative / rule-governed space
#
# In that sense, it functions as an explanatory layer over the verdicts.

# If the goal is to represent nominal meaning in a multi-perspective representational space,
# a dual-space style viewpoint may be more natural.
just_rows = []
for _, row in verdict_df.iterrows():
    focus = row["focus_node"]
    clause_code = row["clause_code"]
    status = row["status"]
    message = row.get("message", None)

    exception_text = None
    if status == "not_applicable":
        matched = exception_df[exception_df["target_clause_code"] == clause_code]
        if len(matched) > 0:
            exception_text = matched.iloc[0]["exception_text"]

    if status == "compliant":
        reason = "No validation issue reported for this clause."
    elif status == "violated":
        reason = message if pd.notna(message) else "Validation violation reported."
    elif status == "warning":
        reason = message if pd.notna(message) else "Validation warning reported."
    elif status == "not_applicable":
        reason = f"Exception applied: {exception_text}"
    elif status == "unknown":
        reason = "Insufficient evidence to distinguish violation from missing information."
    else:
        reason = "Optional / informational clause."

    just_rows.append({
        "focus_node": focus,
        "clause_code": clause_code,
        "status": status,
        "reason": reason,
    })

justification_df = pd.DataFrame(just_rows).sort_values(["focus_node","clause_code"]).reset_index(drop=True)
display(justification_df)

,focus_node,clause_code,status,reason
0,http://example.org/artifact/FD001,5.2.1,compliant,No validation issue reported for this clause.
1,http://example.org/artifact/FD001,5.2.2,compliant,No validation issue reported for this clause.
2,http://example.org/artifact/FD001,5.2.3,compliant,No validation issue reported for this clause.
3,http://example.org/artifact/FD001,5.2.5,informational,Optional / informational clause.
4,http://example.org/artifact/FD002,5.2.1,compliant,No validation issue reported for this clause.
5,http://example.org/artifact/FD002,5.2.2,compliant,No validation issue reported for this clause.
6,http://example.org/artifact/FD002,5.2.3,compliant,No validation issue reported for this clause.
7,http://example.org/artifact/FD002,5.2.5,informational,Optional / informational clause.
8,http://example.org/artifact/FD003,5.2.1,compliant,No validation issue reported for this clause.
9,http://example.org/artifact/FD003,5.2.2,compliant,No validation issue reported for this clause.


## 9. Aggregate the results

In [12]:
print("--- by artifact ---")
display(verdict_df.groupby(["focus_node","status"]).size().unstack(fill_value=0).reset_index())

print("--- by clause ---")
display(verdict_df.groupby(["clause_code","status"]).size().unstack(fill_value=0).reset_index())

--- by artifact ---


status,focus_node,compliant,informational,not_applicable,unknown
0,http://example.org/artifact/FD001,3,1,0,0
1,http://example.org/artifact/FD002,3,1,0,0
2,http://example.org/artifact/FD003,3,1,0,0
3,http://example.org/artifact/FD004,2,1,1,0
4,http://example.org/artifact/FD005,2,1,0,1


--- by clause ---


status,clause_code,compliant,informational,not_applicable,unknown
0,5.2.1,5,0,0,0
1,5.2.2,3,0,1,1
2,5.2.3,5,0,0,0
3,5.2.5,0,5,0,0


## 10. Query the standards graph with SPARQL

In [13]:
query = '''
PREFIX std: <http://example.org/standard/>

SELECT ?clause ?code ?text ?modality ?ref ?cond
WHERE {
  ?clause a std:Clause ;
          std:clauseCode ?code ;
          std:clauseText ?text .
  OPTIONAL { ?clause std:hasModality ?modality . }
  OPTIONAL { ?clause std:referencesClause ?ref . }
  OPTIONAL { ?clause std:hasExceptionCondition ?cond . }
}
ORDER BY ?code
'''

rows = []
for row in g_standard.query(query):
    rows.append(tuple(str(x) if x is not None else None for x in row))

sparql_df = pd.DataFrame(rows, columns=["clause", "code", "text", "modality", "referencesClause", "exceptionCondition"])
display(sparql_df)

,clause,code,text,modality,referencesClause,exceptionCondition
0,http://example.org/standard/Clause_5_2_1,5.2.1,A fire door shall have exactly one identifier.,http://example.org/standard/Shall,None,None
1,http://example.org/standard/Clause_5_2_2,5.2.2,A fire door shall be connected to at least one...,http://example.org/standard/Shall,None,None
2,http://example.org/standard/Clause_5_2_3,5.2.3,A fire door should have at least one inspectio...,http://example.org/standard/Should,None,None
3,http://example.org/standard/Clause_5_2_5,5.2.5,A fire door may have a maintenance note.,http://example.org/standard/May,None,None


## 11. Convert the RDF graph into an edge table for hetero-graph use

In [14]:
artifact_edges_df = graph_to_df(g_artifact).copy()
artifact_edges_df["src"] = artifact_edges_df["subject"]
artifact_edges_df["rel"] = artifact_edges_df["predicate"]
artifact_edges_df["dst"] = artifact_edges_df["object"]
artifact_edges_df = artifact_edges_df[["src","rel","dst"]].sort_values(["src","rel","dst"]).reset_index(drop=True)
display(artifact_edges_df)

,src,rel,dst
0,http://example.org/artifact/FD001,http://example.org/connectedTo,http://example.org/artifact/ZoneA
1,http://example.org/artifact/FD001,http://example.org/hasIdentifier,FD001
2,http://example.org/artifact/FD001,http://example.org/hasInspectionRecord,http://example.org/artifact/Inspect_001
3,http://example.org/artifact/FD001,http://example.org/hasMaintenanceNote,http://example.org/artifact/Note_001
4,http://example.org/artifact/FD001,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/FireDoor
5,http://example.org/artifact/FD002,http://example.org/connectedTo,http://example.org/artifact/ZoneA
6,http://example.org/artifact/FD002,http://example.org/hasIdentifier,FD002-A
7,http://example.org/artifact/FD002,http://example.org/hasIdentifier,FD002-B
8,http://example.org/artifact/FD002,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/FireDoor
9,http://example.org/artifact/FD003,http://example.org/hasIdentifier,FD003


## 12. What makes this sample more advanced than the previous one

1. Separation of `shall / should / may`
2. Exception clauses are represented as a separate graph
3. Cross-references are represented as clause-to-clause relations
4. A separate verdict policy is implemented in addition to validation
5. `not_applicable` and `unknown` are introduced
6. A justification table is constructed
7. An edge table is produced with graph-learning integration in mind
